# Dark-Vessel Detection — train on Colab GPU

Trains the center-heatmap detector on a free/cheap Colab GPU using the tile bundle you uploaded to Google Drive.

**Before running:**
1. Menu **Runtime → Change runtime type → T4 GPU** (or better), Save.
2. On your laptop you ran `python scripts/make_colab_bundle.py` and uploaded `outputs/colab_tiles.tar` to a Drive folder called **`darkvessel`**.
3. Then **Runtime → Run all** and approve the Drive permission popup.

Checkpoints are written to `darkvessel/checkpoints/` **on your Drive**, so they survive a Colab disconnect. When training ends (or even mid-run), download `best.pt` from Drive to your laptop at `outputs/checkpoints/best.pt` and continue locally with `scripts/04_predict_scene.py`.

In [ ]:
# 1. Confirm a GPU is attached (you should see a Tesla T4 / L4 / A100 table)
!nvidia-smi

In [ ]:
# 2. Get the code
!git clone https://github.com/Harsh-Antares/dark-vessel-detection.git
%cd dark-vessel-detection

In [ ]:
# 3. Connect Google Drive (a popup asks for permission)
from google.colab import drive
drive.mount('/content/drive')

BUNDLE = '/content/drive/MyDrive/darkvessel/colab_tiles.tar'
CKPT_DIR = '/content/drive/MyDrive/darkvessel/checkpoints'

import os
assert os.path.exists(BUNDLE), f'Bundle not found at {BUNDLE} — upload colab_tiles.tar to the darkvessel folder on Drive.'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Drive connected, bundle found.')

In [ ]:
# 4. Unpack the tiles onto Colab's fast local disk (~2 minutes)
!mkdir -p outputs
!tar xf {BUNDLE} -C outputs/
!head -2 outputs/tiles/chips_train.csv
!ls outputs/tiles/train | wc -l && ls outputs/tiles/validation | wc -l

In [ ]:
# 5. Train. Checkpoints go straight to Drive.
#    If you hit 'CUDA out of memory', change --batch-size to 4.
!python scripts/03_train.py --batch-size 8 --checkpoints-dir {CKPT_DIR}

## After training

- The best checkpoint (highest validation F1) is at **`Drive → darkvessel → checkpoints → best.pt`**.
- Download it to your laptop and place it at `outputs/checkpoints/best.pt` inside the project.
- Continue locally with full-scene inference:
  ```
  python scripts/04_predict_scene.py --all --split validation
  python scripts/05_dark_split_and_eval.py --split validation
  ```
- If Colab disconnected mid-run: `last.pt` / `best.pt` on Drive are from the most recent completed epochs — often already usable. **Before re-running the notebook, rename them on Drive** (e.g. `best_run1.pt`): a fresh run starts from scratch and will overwrite them with its own early (worse) checkpoints.